## Visualize SHAP Interaction Value in XGBoostLSS

In [ ]:
import os
import pickle
import sys

import numpy as np
import scipy.stats as scst
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

sys.path.append("../")
from helper import load_data, preprocess, data_source_release

sys.path.insert(0, '../../XGBoostLSS-master')
from xgboostlss.model import *
from xgboostlss.distributions.LogNormal import LogNormal
from xgboostlss.distributions.Weibull import Weibull
from xgboostlss.distributions.Exponential import Exponential
from xgboostlss.distributions.InverseGaussian import InverseGaussian


# 1) Select a scenario
SCENARIO = 'clasp_factoring'
# SCENARIO = 'saps-CVVAR'
# SCENARIO = 'lpg-zeno'
# SCENARIO = 'yalsat_qcp'
# SCENARIO = 'spear_qcp'
# SCENARIO = 'spear_swgcp'
# SCENARIO = 'yalsat_swgcp'

# 2) Select a distribution
DIST = 'lognormal'
# DIST = 'invgauss'
# DIST = 'expon'
# DIST = 'weibull'

In [ ]:
# 3) Load data
sc_dict = data_source_release.get_sc_dict()
data_dir = data_source_release.get_data_dir()

runtimes, features, _ = load_data.get_data(
    scenario=SCENARIO, 
    data_dir=data_dir,
    sc_dict=sc_dict, 
    retrieve=sc_dict[SCENARIO]['use']
)
print(runtimes.shape, features.shape)

X_train = np.array(features)
y_train = np.array(runtimes)

# Flatten data
X_train, _ = preprocess.preprocess_features(X_train, X_train, scal="meanstd")
X_trn_flat = np.concatenate([[x for _ in range(100)] for x in X_train])
y_trn_flat = y_train.flatten().reshape([-1, 1])

# Reset seed
np.random.seed(2)

# Min/Max Scale runnningtimes
y_max_ = np.max(y_trn_flat)
y_min_ = 0
y_trn_flat = (y_trn_flat - y_min_) / y_max_
y_train = (y_train - y_min_) / y_max_

# Get data matrix
dtrain = xgb.DMatrix(X_trn_flat, label=y_trn_flat)

In [ ]:
# Default values in NAS configuration space
opt_params = {
    "n_rounds":                 100,
    "booster":                  "gbtree",
    "max_depth":                13,
    "min_child_weight":         39,
    "colsample_bytree":         0.2545374925231651,
    "colsample_bylevel":        0.6909224923784677,
    "lambda":                   31.393252465064943,
    "alpha":                    0.24167936088332426,
    "learning_rate":            0.008237525103357958,
    "n_jobs":                   -1,
}

# 4) Choose a model
if DIST == "lognormal":
    xgblss = XGBoostLSS(
        LogNormal(stabilization="None",  
                    response_fn="exp",      
                    loss_fn="nll"))
elif DIST == "weibull":
    xgblss = XGBoostLSS(
        Weibull(stabilization="None",  
                response_fn="exp",      
                loss_fn="nll"))
elif DIST == "expon":
    xgblss = XGBoostLSS(
        Exponential(stabilization="None",  
                    response_fn="exp",      
                    loss_fn="nll"))
elif DIST == "invgauss":
    xgblss = XGBoostLSS(
        InverseGaussian(stabilization="None",  
                        response_fn="exp",      
                        loss_fn="nll"))
else:
    raise ValueError("Don't know %s" % DIST)


np.random.seed(2)
torch.manual_seed(2)

opt_params = opt_params.copy()
num_boost_round = opt_params["n_rounds"]        
del opt_params["n_rounds"]

xgblss.train(params=opt_params, dtrain=dtrain, num_boost_round=num_boost_round)
dtrain = xgb.DMatrix(X_train, label=y_train)
pred = xgblss.predict(dtrain, pred_type="parameters")

In [ ]:
# 5) Visualize SHAP values
explainer = shap.Explainer(xgblss)
explanation = explainer(dtrain)
shap_values = explanation.values

shap.plots.beeswarm(explanation)